In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt

pd.set_option("display.max_columns", None)

In [ ]:
agg_gb = gpd.read_file("/Users/sonle/Documents/work/ADB/adb-school-optimize/datasets/aggregated_district_data.geojson")

curated =pd.read_csv("/Users/sonle/Documents/work/ADB/adb-school-optimize/datasets/png_curated_sec_schools_access_v3_clean.csv")

df = curated.copy(deep=True)

curated.shape

In [ ]:
# curated.drop_duplicates(subset=["School Name"]).to_csv("/Users/sonle/Documents/work/ADB/adb-school-optimize/datasets/png_curated_sec_schools_access_v3_clean.csv", index=False)

In [ ]:
ranked = pd.read_csv("/Users/sonle/Documents/work/ADB/adb-school-optimize/datasets/png_curated_sec_schools_ranked.csv")
ranked.shape
# ranked.drop_duplicates(subset=["School Name"]).to_csv("/Users/sonle/Documents/work/ADB/adb-school-optimize/datasets/png_curated_sec_schools_ranked.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads

# =========================================================
# HELPERS
# =========================================================
def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    valid = s.dropna()
    if valid.empty:
        return pd.Series(np.nan, index=s.index)
    mn, mx = valid.min(), valid.max()
    if mx == mn:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / (mx - mn)

def inv_minmax(s: pd.Series) -> pd.Series:
    return 1 - minmax(s)

def clip01(s: pd.Series) -> pd.Series:
    return s.clip(lower=0, upper=1)

def clean_text(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower()

def row_mean(df_, cols):
    if all(isinstance(c, str) for c in cols):
        tmp = df_[cols].apply(pd.to_numeric, errors="coerce")
        return tmp.mean(axis=1, skipna=True)
    tmp = pd.concat(cols, axis=1).apply(pd.to_numeric, errors="coerce")
    return tmp.mean(axis=1, skipna=True)

def safe_div(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    out = num / den.replace({0: np.nan})
    return out

def wkt_area_m2(series: pd.Series, source_crs="EPSG:4326", area_crs="EPSG:3857") -> pd.Series:
    geom = series.apply(lambda x: loads(x) if pd.notna(x) else None)
    gs = gpd.GeoSeries(geom, crs=source_crs)
    return gs.to_crs(area_crs).area

# =========================================================
# HIERARCHICAL IMPUTATION
# For numeric: district median -> province median -> national median
# For categorical: district mode -> province mode -> national mode
# =========================================================
numeric_school_cols = [
    "Number of Available Teachers",
    "Total Number of Classrooms",
    "Number of Permanent Classrooms",
    "Number of Semi-Permanent Classrooms",
    "Number of Bush Material Classrooms",
    "Number of Houses for Teachers",
    "Number of Libraries",
    "Number of Workshops",
    "Number of Practical Skills Buildings",
    "Number of Home Economics Buildings",
    "Number of Computer Labs",
    "Number of Specialized Classrooms",
]

categorical_school_cols = [
    "Power Source",
    "Water Source",
    "Toilets",
]

for c in numeric_school_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

def mode_or_nan(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else np.nan

for c in numeric_school_cols:
    district_med = df.groupby(["Province", "District"])[c].transform("median")
    province_med = df.groupby("Province")[c].transform("median")
    national_med = df[c].median()
    df[c + "_orig_missing"] = df[c].isna().astype(int)
    df[c] = df[c].fillna(district_med).fillna(province_med).fillna(national_med)

for c in categorical_school_cols:
    district_mode = df.groupby(["Province", "District"])[c].transform(mode_or_nan)
    province_mode = df.groupby("Province")[c].transform(mode_or_nan)
    national_mode = mode_or_nan(df[c])
    df[c + "_orig_missing"] = df[c].isna().astype(int)
    df[c] = df[c].fillna(district_mode).fillna(province_mode).fillna(national_mode)

# =========================================================
# SCHOOL-LEVEL PROXIES
# =========================================================

# 1) Accessibility / remoteness
Locality_map = {
    "highly accessible": 0.00,
    "accessible": 0.20,
    "moderately accessible": 0.45,
    "remote": 0.70,
    "very remote": 0.90,
    "extremely remote": 1.00
}
df["Locality_clean"] = df["Locality"].map(clean_text)
df["Locality_score"] = df["Locality_clean"].map(Locality_map)
df["Locality_score"] = df["Locality_score"].fillna(inv_minmax(df["pop_with_access_walking"]))

# Access opportunity counts are "higher is better", so invert.
df["walk_access_deficit"] = inv_minmax(df["pop_with_access_walking"])
df["cycle_access_deficit"] = inv_minmax(df["pop_with_access_cycling"])
df["drive_access_deficit"] = inv_minmax(df["pop_with_access_driving"])
df["school_access_barrier"] = (
    0.50 * df["walk_access_deficit"] +
    0.20 * df["cycle_access_deficit"] +
    0.30 * df["drive_access_deficit"]
)
df["school_access_barrier"] = clip01(df["school_access_barrier"])

# 2) Teacher scarcity proxy
df["teacher_scarcity"] = inv_minmax(df["Number of Available Teachers"])

# 3) Classroom adequacy / quality proxies
df["perm_class_share"] = clip01(
    safe_div(df["Number of Permanent Classrooms"], df["Total Number of Classrooms"])
)
df["semi_class_share"] = clip01(
    safe_div(df["Number of Semi-Permanent Classrooms"], df["Total Number of Classrooms"])
)
df["bush_class_share"] = clip01(
    safe_div(df["Number of Bush Material Classrooms"], df["Total Number of Classrooms"])
)

# fewer classrooms = worse
df["classroom_stock_deficit"] = inv_minmax(df["Total Number of Classrooms"])

# 4) Facility breadth / learning infrastructure
facility_cols = [
    "Number of Libraries",
    "Number of Workshops",
    "Number of Practical Skills Buildings",
    "Number of Home Economics Buildings",
    "Number of Computer Labs",
    "Number of Specialized Classrooms",
]
for c in facility_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["facility_count_total"] = df[facility_cols].sum(axis=1)
df["facility_deficit"] = inv_minmax(df["facility_count_total"])

# 5) Teacher housing
df["teacher_housing_deficit"] = inv_minmax(df["Number of Houses for Teachers"])

# 6) Service readiness from text recoding
def score_power(x):
    x = clean_text(x)
    if pd.isna(x):
        return np.nan
    if any(k in x for k in ["none", "no power", "nil", "not available"]):
        return 1.0
    if "generator" in x:
        return 0.7
    if "solar" in x and "grid" not in x:
        return 0.4
    if any(k in x for k in ["png power", "grid", "mains", "electricity"]):
        return 0.2
    return 0.5

def score_water(x):
    x = clean_text(x)
    if pd.isna(x):
        return np.nan
    if any(k in x for k in ["river", "stream", "creek", "pond", "unprotected", "not available", "none"]):
        return 1.0
    if any(k in x for k in ["well", "bore", "rain", "tank"]):
        return 0.5
    if any(k in x for k in ["piped", "tap", "reticulated"]):
        return 0.2
    return 0.5


df["power_deficit"] = df["Power Source"].map(score_power).fillna(0.5)
df["water_deficit"] = df["Water Source"].map(score_water).fillna(0.5)

df["service_deficit"] = (
    0.40 * df["power_deficit"] +
    0.30 * df["water_deficit"] 
)

# =========================================================
# SCHOOL NEED (S)
# No enrollment is available, so use proxy-based school need.
# =========================================================
df["S"] = (
    0.25 * df["Locality_score"] +
    0.20 * df["school_access_barrier"] +
    0.15 * df["teacher_scarcity"] +
    0.20 * df["classroom_stock_deficit"] +
    0.15 * df["service_deficit"] +
    0.05 * row_mean(df, ["facility_deficit", "teacher_housing_deficit"])
)
df["S"] = clip01(df["S"])

# =========================================================
# DISTRICT-LEVEL CONTEXT (A)
# =========================================================
df["adm_walk_deficit"] = inv_minmax(df["Access Walking (%)"])
df["adm_drive_deficit"] = inv_minmax(df["Access Driving (%)"])
df["adm_cycle_deficit"] = inv_minmax(df["Access Cycling (%)"])
df["AccAdm"] = (
    0.50 * df["adm_walk_deficit"] +
    0.20 * df["adm_cycle_deficit"] +
    0.30 * df["adm_drive_deficit"]
)

df["fixed_down_deficit"] = inv_minmax(df["Fixed Broadband Download Speed (MB/s)"])
df["mobile_down_deficit"] = inv_minmax(df["Mobile Internet Download Speed (MB/s)"])
df["ServAdm"] = 0.5 * df["fixed_down_deficit"] + 0.5 * df["mobile_down_deficit"]

df["prog10_deficit"] = inv_minmax(df["Rate of Grade 7 who progressed to Grade 10 (%)"])
df["sec_participation_deficit"] = inv_minmax(df["Grade 7-10 Students per 1000 Population"])

df["female_share_7_12"] = safe_div(df["Female students grade 7-12"], df["Total enrollment Grade 7-12"])
df["female_disadvantage"] = clip01((0.5 - df["female_share_7_12"]) / 0.5)
df["female_disadvantage"] = df["female_disadvantage"].fillna(0)

df["ntl_deficit"] = inv_minmax(df["Total Nighttime Luminosity"])
df["SocioAdm"] = (
    0.45 * df["ntl_deficit"] +
    0.35 * df["sec_participation_deficit"] +
    0.20 * inv_minmax(df["Secondary students per 1000 people"])
)

df["conflict_events_n"] = minmax(df["Conflict Events"])
df["conflict_fatalities_n"] = minmax(df["Conflict Fatalities"])
df["conflict_exposure_n"] = minmax(df["Conflict Population Exposure"])
df["ConflictAdm"] = (
    0.30 * df["conflict_events_n"] +
    0.30 * df["conflict_fatalities_n"] +
    0.40 * df["conflict_exposure_n"]
)

df["A"] = (
    0.30 * df["AccAdm"] +
    0.20 * df["ServAdm"] +
    0.25 * df["prog10_deficit"] +
    0.15 * df["SocioAdm"] +
    0.10 * df["ConflictAdm"]
)
df["A"] = clip01(df["A"])

# =========================================================
# PHYSICAL / RESILIENCE
# Flood R comes from your flood processing.
# lc_landterr_score comes from your land-cover processing.
# =========================================================
df["R_phys"] = (
    0.70 * df["R"].fillna(df["R"].median()) +
    0.30 * df["lc_landterr_score"].fillna(df["lc_landterr_score"].median())
)
df["R_phys"] = clip01(df["R_phys"])

# =========================================================
# GIRLS BONUS (bounded)
# =========================================================
df["G"] = np.minimum(
    0.08,
    0.05 * df["female_disadvantage"] +
    0.03 * df["Locality_score"]
)

# =========================================================
# STAGE 1 NEED SCORE
# =========================================================
df["Need"] = (
    0.55 * df["S"] +
    0.25 * df["A"] +
    0.20 * df["R_phys"] +
    df["G"]
)
df["Need"] = clip01(df["Need"])

# =========================================================
# IMPACT POTENTIAL (I)
# =========================================================
df["accessible_pop"] = row_mean(df, [
    "pop_with_access_walking",
    "pop_with_access_cycling",
    "pop_with_access_driving"
])

# Your cachment_area_* fields are WKT polygons, not numeric area values.
# Convert them to projected area first.
df["cachment_area_walking_m2"] = wkt_area_m2(df["cachment_area_walking"])
df["cachment_area_cycling_m2"] = wkt_area_m2(df["cachment_area_cycling"])
df["cachment_area_driving_m2"] = wkt_area_m2(df["cachment_area_driving"])

df["catchment_area"] = row_mean(df, [
    "cachment_area_walking_m2",
    "cachment_area_cycling_m2",
    "cachment_area_driving_m2"
])

df["accessible_pop_n"] = minmax(df["accessible_pop"])
df["catchment_area_n"] = minmax(df["catchment_area"])

df["school_gap"] = (
    0.45 * df["service_deficit"] +
    0.30 * df["classroom_stock_deficit"] +
    0.25 * df["teacher_scarcity"]
)

df["I"] = (
    0.45 * df["accessible_pop_n"] +
    0.25 * df["catchment_area_n"] +
    0.30 * df["school_gap"]
)
df["I"] = clip01(df["I"])

# =========================================================
# PRACTICALITY (P)
# =========================================================
df["P"] = (
    0.50 * (1 - df["lc_landterr_score"].fillna(df["lc_landterr_score"].median())) +
    0.50 * (1 - df["R"].fillna(df["R"].median()))
)
df["P"] = clip01(df["P"])

# =========================================================
# FINAL PRIORITY SCORE
# =========================================================
df["Priority"] = (
    0.70 * df["Need"] +
    0.20 * df["I"] +
    0.10 * df["P"]
)
df["Priority"] = clip01(df["Priority"])

# =========================================================
# DATA CONFIDENCE
# =========================================================
core_missing_flags = [c for c in df.columns if c.endswith("_orig_missing")]
if core_missing_flags:
    df["data_confidence"] = 1 - df[core_missing_flags].mean(axis=1)
else:
    df["data_confidence"] = 1.0

# =========================================================
# OPTIONAL: STAGE-1 SCREENING / BALANCING
# =========================================================
need_cutoff = df["Need"].quantile(0.65)
df["stage1_selected"] = (df["Need"] >= need_cutoff).astype(int)

# =========================================================
# FINAL OUTPUT
# =========================================================
rank_cols = [
    "School Name", "Province", "District",
    "S", "A", "R_phys", "G", "Need", "I", "P", "Priority",
    "data_confidence", "stage1_selected",
    "Locality_score", "school_access_barrier", "teacher_scarcity",
    "classroom_stock_deficit",
    "service_deficit", "facility_deficit", "teacher_housing_deficit",
    "prog10_deficit", "female_disadvantage", "ConflictAdm",
    "R", "lc_landterr_score",
    "accessible_pop", "catchment_area"
]

df_ranked = df.sort_values(["Priority", "Need"], ascending=[False, False]).copy()


In [ ]:
df_ranked = df_ranked[['School Name', 'S', 'A', 'R_phys', 'I', 'P', "Need", "Priority"]]
for col in df_ranked.columns[1:]:
    df_ranked[col] = df_ranked[col].round(4) * 100
df_ranked

In [ ]:
df_ranked.to_csv(r"D:\Work\ADB\adb-school-optimize\data/datasets\png_curated_sec_schools_ranked.csv", index=False)